# HW4 — Regional temporal variability (historical–SSP2-4.5 **r25**)

MPI-ESM1-2-LR **r25** (`tas`), regional means over **Arctic (60–90°N)** and **Tropics (20°S–20°N)**:

1. **Linear trends** (annual means), least-squares slope → **°C per decade**, for **1850–2100**, **1900–2000**, **2000–2100**.
2. **Standard deviations** of **annual-mean** regional temperature over the same three periods.
3. **Mean seasonal cycle per calendar decade**: for each decade (1850s … 2100s), monthly climatology of the regional mean; shown as **month × decade** maps.

All figures are written to **one PDF** at the end.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

STUDENT_NAME = "Nasrul Huda"
BASE = "/Users/nayz/Documents/UHH/climate_modelling"

HIST_R25_FILE = f"{BASE}/model_outputs/tas_Amon_MPI-ESM1-2-LR_historical-ssp245_r25i1p1f1_g025_185001-210012.nc"

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
hist_r25 = xr.open_dataset(HIST_R25_FILE, decode_times=time_coder)["tas"]
hist_r25_y = hist_r25.groupby("time.year").mean("time")


def regional_weighted_mean(da, lon_min, lon_max, lat_min, lat_max):
    sub = da.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))
    w = np.cos(np.deg2rad(sub["lat"]))
    return sub.weighted(w).mean(("lat", "lon"))


regions = {
    "Arctic (60N-90N)": dict(lon_min=0, lon_max=360, lat_min=60, lat_max=90),
    "Tropics (20S-20N)": dict(lon_min=0, lon_max=360, lat_min=-20, lat_max=20),
}

In [ ]:
periods = [
    ("1850–2100", 1850, 2100),
    ("1900–2000", 1900, 2000),
    ("2000–2100", 2000, 2100),
]

region_list = list(regions.items())
n_reg = len(region_list)
n_per = len(periods)

trend_table = np.full((n_per, n_reg), np.nan)
std_table = np.full((n_per, n_reg), np.nan)

for i, (_, y0, y1) in enumerate(periods):
    for j, (_, box) in enumerate(region_list):
        ts = regional_weighted_mean(hist_r25_y, **box) - 273.15
        ts = ts.sel(year=slice(y0, y1))
        y = ts["year"].values.astype(float)
        v = ts.values.astype(float)
        m = np.isfinite(v)
        if m.sum() >= 2:
            slope, _ = np.polyfit(y[m], v[m], 1)
            trend_table[i, j] = slope * 10.0
        std_table[i, j] = float(ts.std("year"))

# --- Page 1: linear trends (°C / decade) ---
fig_trend, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
x = np.arange(n_per)
width = 0.36
colors = ["#2c5aa0", "#c44e52"]
for j, (name, _) in enumerate(region_list):
    offset = (j - (n_reg - 1) / 2) * width
    ax.bar(
        x + offset,
        trend_table[:, j],
        width,
        label=name.split(" (")[0],
        color=colors[j % len(colors)],
        edgecolor="0.2",
        linewidth=0.5,
    )

ax.axhline(0, color="0.4", lw=0.8, ls="--")
ax.set_xticks(x)
ax.set_xticklabels([p[0] for p in periods])
ax.set_ylabel("Linear trend (°C / decade)")
ax.set_xlabel("Period")
ax.set_title(
    f"Annual-mean temperature trends — MPI-ESM1-2-LR historical–SSP2-4.5 r25 — {STUDENT_NAME}"
)
ax.legend(title="Region")
ax.grid(True, axis="y", alpha=0.3)

# --- Page 2: standard deviations ---
fig_std, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
for j, (name, _) in enumerate(region_list):
    offset = (j - (n_reg - 1) / 2) * width
    ax.bar(
        x + offset,
        std_table[:, j],
        width,
        label=name.split(" (")[0],
        color=colors[j % len(colors)],
        edgecolor="0.2",
        linewidth=0.5,
    )

ax.set_xticks(x)
ax.set_xticklabels([p[0] for p in periods])
ax.set_ylabel("Std. dev. of annual-mean tas (°C)")
ax.set_xlabel("Period")
ax.set_title(
    f"Interannual variability (σ of annual means) — historical–SSP2-4.5 r25 — {STUDENT_NAME}"
)
ax.legend(title="Region")
ax.grid(True, axis="y", alpha=0.3)

# --- Page 3: mean seasonal cycle per calendar decade (monthly climatology per decade) ---


def seasonal_cycle_matrix_monthly(ts_m):
    """Return (decades_sorted, Z) with shape (n_decades, 12) for regional monthly series."""
    y = ts_m["time"].dt.year.values
    dvec = (y // 10) * 10
    decades = np.unique(dvec)
    mat = np.full((len(decades), 12), np.nan)
    for di, d in enumerate(decades):
        sel = ts_m.sel(time=np.logical_and(y >= d, y < d + 10))
        if sel.sizes.get("time", 0) == 0:
            continue
        clim = sel.groupby("time.month").mean("time")
        for mo in range(1, 13):
            if mo in clim["month"].values:
                mat[di, mo - 1] = float(clim.sel(month=mo))
    return decades, mat


fig_seas, axes = plt.subplots(
    1, 2, figsize=(13.5, 5.8), constrained_layout=True, sharey=True
)
month_edges = np.arange(0.5, 13.5, 1)

for ax, (reg_name, box) in zip(axes, region_list):
    ts_m = regional_weighted_mean(hist_r25, **box) - 273.15
    decades, Z = seasonal_cycle_matrix_monthly(ts_m)
    dec_edges = np.append(decades.astype(float), float(decades[-1]) + 10.0)
    pcm = ax.pcolormesh(
        month_edges,
        dec_edges,
        Z,
        shading="flat",
        cmap="RdYlBu_r",
    )
    ax.set_xlabel("Calendar month")
    ax.set_ylabel("Decade (start year)")
    ax.set_title(reg_name.split(" (")[0])
    ax.set_xticks(np.arange(1, 13))

fig_seas.suptitle(
    f"Mean seasonal cycle by decade (°C) — historical–SSP2-4.5 r25 — {STUDENT_NAME}",
    fontsize=12,
)
cbar = fig_seas.colorbar(pcm, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
cbar.set_label("tas (°C)")

# --- Console summary ---
print("Linear trend (°C / decade):")
print(
    "Period".ljust(18)
    + "".join(r[0].split(" (")[0][:12].ljust(14) for r in region_list)
)
for i, pl in enumerate(periods):
    print(
        pl[0].ljust(18)
        + "".join(f"{trend_table[i, j]:.4f}".ljust(14) for j in range(n_reg))
    )
print()
print("Std. dev. (°C) of annual-mean tas:")
print(
    "Period".ljust(18)
    + "".join(r[0].split(" (")[0][:12].ljust(14) for r in region_list)
)
for i, pl in enumerate(periods):
    print(
        pl[0].ljust(18)
        + "".join(f"{std_table[i, j]:.4f}".ljust(14) for j in range(n_reg))
    )

out_pdf = f"{BASE}/HW4_regional_temporal_variability.pdf"
with PdfPages(out_pdf) as pdf:
    pdf.savefig(fig_trend)
    pdf.savefig(fig_std)
    pdf.savefig(fig_seas)

print()
print("Saved 3-page PDF:", out_pdf)